In [1]:
import pyaudio
import soundfile as sf
import time
import io
import numpy as np
import torch
import faiss
import torch.nn as nn
from tqdm import tqdm

In [2]:
torch.cuda.is_available()

False

In [3]:
SAMPLE_RATE=16000
CHUNK_SIZE=1024


test_file = "download/amicorpus/ES2011a/audio/ES2011a.Mix-Headset.wav"


def read_from_buffer(buffer, chunk_size):
    chunk = buffer.read(chunk_size)
    return chunk


def stream_audio_from_file(filename, max_chunks=56250):
    if max_chunks is None:
        max_chunks = 56250
    data, samplerate = sf.read(filename, dtype='int16')
    if len(data.shape) > 1:
        data = data[:, 0] # Take only the first channel
        
    audio_buffer = io.BytesIO(data.tobytes())
    num_chunks = len(data) // CHUNK_SIZE
    for i in range(min(num_chunks, max_chunks)):
        try:
            chunk = read_from_buffer(audio_buffer, CHUNK_SIZE * 2)
            if len(chunk) == 0:
                break
            # This mimics the true passage of time
            #time.sleep(CHUNK_SIZE / SAMPLE_RATE)
            input_ = np.frombuffer(chunk, dtype=np.int16) / 32768
            yield input_
        except KeyboardInterrupt:
            pass
        

In [6]:
def make_index(data, num_coarse_clusters=100, m=8, bits_per_code=8, dim=1024, num_points=1500000):
    quantizer = faiss.IndexFlatL2(dim)  # this remains the same
    index = faiss.IndexIVFPQ(quantizer, dim, num_coarse_clusters, m, bits_per_code) # 8 specifies that each sub-vector is encoded as 8 bits
    if data is None:
        data = np.random.random((1500000, dim)).astype('float32') - 0.5
    index.train(data)
    index.add(data)
    return index

In [4]:
def streaming_nn(fn, index, test_file, nprobe=2, topk=4, max_chunks=4):
    index.nprobe = nprobe # let's just search two clusters for now.
    for d in tqdm(stream_audio_from_file(test_file, max_chunks=max_chunks)):
        chunk_data = torch.tensor(d, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
        feats = fn(chunk_data)
        _, _, R = index.search_and_reconstruct(feats.squeeze(0).detach().numpy(), topk)
        #print(R[0, 0, 0:10])

In [7]:
W = nn.Linear(1, 1024)
index = make_index(None, num_coarse_clusters=100, num_points=50000)

In [168]:
"streaming_nn(W, index, test_file, topk=1, max_chunks=None)

198it [00:36,  5.39it/s]


KeyboardInterrupt: 